# Qdislib — The cutting workflow with PyCOMPSs

`gate_cutting` / `wire_cutting` do everything in one call. The
**recommended workflow** breaks that into three steps you control:

1. **generate** the subcircuits,
2. **evaluate** each one (locally, distributed, or on a QPU),
3. **reconstruct** the original expectation value.

This is what you want for distributed runs and for inspecting the pieces.

In [ ]:
import pycompss.interactive as ipycompss
ipycompss.start(graph=True, monitor=1000)  # debug=True, trace=True

In [ ]:
from qibo import models, gates


def example_circuit():
    """A small 5-qubit circuit: cuttable (CZ chain), non-trivial <ZZZZZ> ~= 0.44."""
    circuit = models.Circuit(5)
    circuit.add(gates.RY(0, 0.8))
    circuit.add(gates.CZ(0, 1))
    circuit.add(gates.RY(1, 0.5))
    circuit.add(gates.CZ(1, 2))
    circuit.add(gates.RY(2, 0.6))
    circuit.add(gates.CZ(2, 3))
    circuit.add(gates.RY(3, 0.4))
    circuit.add(gates.CZ(3, 4))
    circuit.add(gates.RX(4, 0.3))
    return circuit


circuit = example_circuit()
print(circuit.draw())

In [ ]:
import Qdislib as qd

## 1. Generate the subcircuits

`find_cut` picks the cut; `gate_cutting_subcircuits` produces the pieces.

In [ ]:
circuit = example_circuit()
cut = qd.find_cut(circuit, gate_cut=True, wire_cut=False, seed=0)  # seed: same cut every run
print("cut:", cut)

subcircuits = qd.gate_cutting_subcircuits(example_circuit(), cut, software="qibo")
print("number of subcircuits:", len(subcircuits))

## 2. Evaluate each subcircuit

`expectation_value` runs one subcircuit and returns its expectation. This
is the loop you'd replace to dispatch to a cluster or a QPU.

In [ ]:
values = [qd.expectation_value(s, shots=8192) for s in subcircuits]

## 3. Reconstruct

The object returned by `*_subcircuits` remembers the cut bookkeeping, so
`reconstruct` needs nothing else.

In [ ]:
result = subcircuits.reconstruct(values)
print("reconstructed:", result)
print("analytical   :", qd.analytical_solution(example_circuit(), "ZZZZZ"))

## Visualise the cut

`draw_cut` shows the circuit DAG with the cut highlighted.

In [ ]:
qd.draw_cut(example_circuit(), cut)  # needs matplotlib; opens a figure

## 4. The same pieces — on real hardware (MareNostrum Ona)

Step 2 is the only thing that changes. The subcircuits are ordinary objects, so
the loop that evaluated them on CPU can dispatch them anywhere — a cluster, a
GPU, or a **QPU**.

At BSC the classical (PyCOMPSs) job and the quantum-control job run on
**separate clusters**, so Qdislib hands circuits over through a shared
directory (`Qdislib.utils.qpu_bridge`): the workflow writes `circuit_<id>.pkl`,
a **listener** on the quantum cluster runs it on the QPU and writes
`result_<id>.pkl` back.

We use a **wire cut** here, and the reason is worth knowing. Gate cutting's
six-term decomposition includes configurations that *measure the cut qubit
mid-circuit and re-prepare it* — and a QPU returns only aggregated final counts,
so the per-shot cut-point outcome is lost. A wire cut measures at the **end** of
a fragment, so every piece is a plain unitary and all of them run on hardware.

The cell below touches no hardware: it loads values **captured on Ona
beforehand**. The reconstruction happens live, here.

In [ ]:
import json

# The Ona data and helper live in a hidden folder, so the file browser shows
# only the notebooks. Nothing here depends on the working directory being exact.
import pathlib
import sys

QCE26_DATA = next(
    (p for p in (pathlib.Path(".qce26"), pathlib.Path(".")) if (p / "ona_results.json").exists()),
    pathlib.Path(".qce26"),
)
if str(QCE26_DATA) not in sys.path:
    sys.path.insert(0, str(QCE26_DATA))

record = json.loads((QCE26_DATA / "ona_results.json").read_text())

print("source   :", record["source"])          # says PLACEHOLDER if not yet captured
print("captured :", record["captured_utc"])
print("cut      :", record["cut"], " shots:", record["shots"])
print()

# Rebuild the same sixteen fragments the capture used (same circuit, same seed).
cut_ona = qd.find_cut(example_circuit(), gate_cut=False, wire_cut=True, seed=0)
subcircuits_ona = qd.wire_cutting_subcircuits(
    example_circuit(), cut_ona, observables="ZZZZZ", software="qibo"
)
assert [list(c) for c in cut_ona] == record["cut"], "cut drifted from the captured one"

print(f"{'#':>2}  {'physical':<10}  {'obs':<4}  {'simulator':>10}  {'on Ona':>10}")
for frag in record["fragments"]:
    print(f"{frag['index']:>2}  {str(frag['physical_qubits']):<10}  {frag['observable']:<4}"
          f"  {frag['simulator']:>+10.4f}  {frag['value']:>+10.4f}")

on_qpu = sum(1 for f in record["fragments"] if f["device"] == "ona")
print(f"\n{on_qpu} of {len(record['fragments'])} fragments ran on the QPU"
      f"  (physical qubits: {record['qpu']['physical_qubits']})")


Now reconstruct — the same call as step 3, fed with the hardware numbers:

In [ ]:
ona_values = [f["value"] for f in record["fragments"]]

print(f"reconstructed from hardware : {subcircuits_ona.reconstruct(ona_values):+.4f}")
print(f"reconstructed in simulation : "
      f"{subcircuits_ona.reconstruct([f['simulator'] for f in record['fragments']]):+.4f}")
print(f"analytical                  : {record['analytical']:+.4f}")

**Cutting does not remove noise.** Each fragment carries readout and gate error,
and reconstruction combines sixteen of them with the quasiprobability weights of
an 8-term decomposition — which amplifies that error rather than averaging it
away. The hardware value comes out visibly below the exact one. Cutting buys you
*width*: a circuit wider than the device, at the price of noise and
exponentially many fragments.

**Every fragment ran on the QPU**, because a wire cut leaves no mid-circuit
measurement. Compare with gate cutting, where four of twelve configurations
would have no hardware answer at all. When you need a *gate* cut on real
hardware, `gate_cutting_hadamard` is the way: it replaces the mid-circuit
measurement with a local ancilla.

The fragments were placed on physical qubits **4, 5 and 6** — chosen from the
day's calibration (CZ fidelities 0.958 and 0.950). Picking qubits matters: on
the same chip that day, three other qubits had readout stuck high.

## Wire cutting — same three steps

Only the generator function changes.

In [ ]:
circuit = example_circuit()
wire_cut = qd.find_cut(circuit, gate_cut=False, wire_cut=True)
subcircuits = qd.wire_cutting_subcircuits(example_circuit(), wire_cut, software="qibo")
values = [qd.expectation_value(s, shots=8192) for s in subcircuits]
print("wire reconstruction:", subcircuits.reconstruct(values))

**Gate vs wire.** Gate cutting splits a two-qubit gate (cost grows fast
with the number of cuts); wire cutting severs a qubit wire. `find_cut`
can weigh both.

## 5. Optional — send one fragment to Ona, live

⚠️ **Only run this when the instructor says the listener is up.** Ona is a
single, real, shared device: this cell is one hardware execution and the whole
room shares the queue. Submit once; don't loop.

`ona_submit.run` handles three things the bridge does not do for you: it ships
the fragment as **QASM text** (MN5 and the quantum cluster run different qibo
versions, so a pickled circuit is not safe to send), it places the fragment on
**known-good physical qubits**, and it marginalises the counts back down. The
wait is bounded at 5 minutes, so a listener that is down fails your cell rather
than freezing it.

In [ ]:
SUBMIT_TO_ONA = False  # <- flip to True only when the instructor says so

if SUBMIT_TO_ONA:
    import os

    import ona_submit  # found via QCE26_DATA on sys.path

    # Writer and listener must agree on this directory. The instructor gives it.
    os.environ["QDISLIB_SUBCIRCUITS_DIR"] = "/gpfs/projects/nct_386/qce26/subcircuits"

    fragment = subcircuits_ona[0]

    # Under PyCOMPSs the fragment is a *future*, so it cannot be inspected here.
    # ona_submit reads it inside a task, which is where the object is real.
    qasm, observable, nqubits, _ = ona_submit.describe(fragment)
    print("hand-off directory:", ona_submit.spool_dir())
    print(f"{nqubits}-qubit fragment, observable {observable}:")
    print(qasm)

    try:
        value, counts = ona_submit.run(fragment, timeout=300)
        print("counts from Ona:", counts)
        print(f"<{observable}> on Ona    : {value:+.4f}")
        print(f"<{observable}> simulated : "
              f"{qd.expectation_value(fragment, shots=8192):+.4f}")
    except qd.QPUTimeoutError as exc:
        print("No result in time — is the listener running?\n ", exc)
    except qd.QPUExecutionError as exc:
        print("The QPU could not run this circuit:\n ", exc)
else:
    print("Skipped. Set SUBMIT_TO_ONA = True to send one fragment to Ona.")


**The supported one-liner** (instructor demo — left commented on purpose).
`qpu_dict={"MN_Ona": 2}` means *"use the QPU for anything that fits in 2
qubits"*, so the six 2-qubit fragments go to Ona from inside their own PyCOMPSs
tasks while the 3-qubit ones stay on CPU — a genuinely hybrid CPU+QPU run. It is
**six** hardware executions per call, so it is not something a full room runs at
once.

```python
value = qd.gate_cutting(example_circuit(), cut_ona, observables="ZZZZZ",
                        software="qibo", qpu=True, qpu_dict={"MN_Ona": 2})
```

In [ ]:
ipycompss.stop(sync=True)

In [ ]:
ipycompss.complete_task_graph(fit=True)